In [ ]:
import pandas as pd

df = pd.read_csv("../data/creditcard.csv")
print(df.shape)
print(df['Class'].value_counts())
print(df.head(3))

In [ ]:
#find null values
df.isnull().sum().max()

In [ ]:
# find percentage of fraud and not fraud class in the data
print('No Frauds', round(df['Class'].value_counts()[0]/len(df) * 100,2), '% of the dataset')
print('Frauds', round(df['Class'].value_counts()[1]/len(df) * 100,2), '% of the dataset')

In [ ]:
# Visualize the Class values 
import matplotlib.pyplot as plt
import seaborn as sns
colors = ["#0101DF", "#DF0101"]

sns.countplot(x='Class', data=df, palette=colors)
plt.title('Class Distributions \n (0: No Fraud || 1: Fraud)', fontsize=14)

In [ ]:
# The target column is imbalanced
df['Class'].value_counts(normalize=True)

In [ ]:
import pandas as pd

# 1. All fraud rows
fraud = df[df['Class'] == 1]

# 2. Same number of non-fraud rows, sampled randomly
non_fraud = df[df['Class'] == 0].sample(n=len(fraud), random_state=42)

# 3. Combine them into one smaller, balanced dataset
df_small = pd.concat([fraud, non_fraud], axis=0)

fraud_small = df_small[df_small['Class'] == 1]
nonfraud_small = df_small[df_small['Class'] == 0]

In [ ]:
# checking to see if Amount column is an important feature in identifying Class
# here we see that the bars almost overlap, so Amount is not a strong indicator
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
fraud_small['Amount'].hist(bins=30)
plt.title('Fraud: Amount')

plt.subplot(1, 2, 2)
nonfraud_small['Amount'].hist(bins=30)
plt.title('Non-Fraud: Amount')

plt.show()

In [ ]:
# finding which features gives the most strong signal when compared between fraud vs non-fraud class
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # helps with the imbalance
)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
feature_importance = pd.Series(importances, index=X.columns).sort_values(ascending=False)
print(feature_importance.head(10))

In [ ]:
# checking to see the overlap with V14 column 
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
fraud_small['V14'].hist(bins=30)
plt.title('Fraud: V14')

plt.subplot(1, 2, 2)
nonfraud_small['V14'].hist(bins=30)
plt.title('Non-Fraud: V14')

plt.show()

#shows that V14 is an important feature in identifying fraud from non-fraud 

In [ ]:
# Train a model to create predictions 
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

In [ ]:
# Create a Confusion Matrix from the predicted values
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=['Actual 0 (non-fraud)', 'Actual 1 (fraud)'],
    columns=['Pred 0 (non-fraud)', 'Pred 1 (fraud)']
)
print(cm_df)

In [ ]:
# Our model is able to predict Non - Fraud values with high percentage compared to Fraud Values
from sklearn.metrics import accuracy_score, classification_report

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))